# Setup

`nhp_data` is a python package.
So to be able to access the functions inside it, first need to install it:

In [0]:
%pip install nhp_data

Then loading in functions:

In [0]:
from pyspark.sql import functions as F
from nhp.data.get_spark import get_spark
from nhp.data.table_names import table_names
from nhp.data.inputs_data.ip.rates import get_population
from nhp.data.inputs_data.direct_standardisation import directly_standardise
from nhp.data.inputs_data.helpers import complete_age_sex_rows
from nhp.data.raw_data.mitigators.ip.hes_datasets import (
    any_diagnosis,
    #any_procedure,
    nhp_apc,
    primary_diagnosis,
    #primary_procedure,
    #secondary_diagnosis,
)
from pyspark.sql.window import Window

Creating a function to apply exclusion criteria on both prior and readmission spells:

# Updating existing readmissions TPMA to new definition

In [0]:
def apply_readmissions_exclusions(data):
    data = data.alias("data")

    cancer_spells = (
        spark.read.table(table_names.hes_apc)
        .filter(F.col("last_episode_in_spell"))
        .join(spark.read.table(table_names.hes_apc_diagnoses), ["epikey", "fyear", "procode3"], "left")
        .filter(
            (F.col("diagnosis") == "Z511") |
            (F.left(F.col("diagnosis"), F.lit(1)) == "C") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D37") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D38") |
            (F.left(F.col("diagnosis"), F.lit(3)) == "D39") |          
            (F.left(F.col("diagnosis"), F.lit(2)) == "D4")             
            )
        .select(
            F.col("person_id_deid").alias("cancer_person_id"), 
            F.col("disdate").alias("cancer_disdate")
            )
        .distinct()
        .alias("cancer_spells")
        )

    filtered = (
        data
        # remove if cancer diagnosis in last 365 days
        .withColumn("365_days_before_disdate", F.date_add(F.col("disdate"), -365))
        .join(
            cancer_spells,
            (F.col("data.person_id") == F.col("cancer_spells.cancer_person_id")) &
            (F.col("cancer_spells.cancer_disdate") <= F.col("data.disdate")) &
            (F.col("cancer_spells.cancer_disdate") >= F.col("365_days_before_disdate")),
            "leftanti"
        )
        # remove obstetrics
        .filter(
            ~((F.col("tretspef") == "501") & F.col("primary_diagnosis").rlike("O")) &
            ~((F.col("tretspef") == "560") & F.col("primary_diagnosis").rlike("O")) & 
            ~((F.col("tretspef") == "610") & F.col("primary_diagnosis").rlike("O"))
            )
        # remove MH services
        .filter(~F.col("tretspef").isin("656", "700", "710", "711", "712", "713", "715", "720", "721", "722", "723", "724", "725", "726", "727", "730"))
        # remove transfers in and out
        .filter(~F.col("admimeth").isin("2B", "28", "81"))
        .filter(~F.col("disdest").isin("49", "50", "51", "52", "53", "84", "87"))
        # exclude day cases and regular attendances
        .filter(F.col("epitype") == 1)
        .filter(F.col("classpat") == 1)
        )
    return(filtered)

In the chunk below

- I copied and pasted the readmissions function from <https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/raw_data/mitigators/ip/activity_avoidance/readmission_within_28_days.py>.
- Tidied some bits so it will run here.
- Tweaked the definition to within 30 days (defined as `between(0, 29)`) and renamed function.
- Used `apply_readmissions_exclusions()` on both `prior` and `readm`. To do this:
    - Joined `prior` to `hes_apc_diagnoses` to get `primary_diagnosis`.
    - Joined `readm` back to `hes_apc` to get `disdest`.

In [0]:
def get_readmissions_within_30_days():
    spark = get_spark()

    readm = apply_readmissions_exclusions(
        nhp_apc
        # joining back to hes_apc to get disdest column - may be better to add disdest column to nhp_apc in longer run
        .join(
            spark.read.table(table_names.hes_apc).select(F.col("epikey"), F.col("fyear"), F.col("procode3"), F.col("disdest")), 
            ["epikey", "fyear", "procode3"],
            "left"
            )
        ).alias("readm")

    # make sure to use full hes table - our nhp views filter on certain columns
    # (e.g. not all providers included)
    prior = (
        spark.read.table(table_names.hes_apc)
        .filter(F.col("last_episode_in_spell"))
        # only spells discharged on clinical advice
        .filter(F.col("dismeth").isin(1, 3))
        # remove well babies
        .filter(F.col("well_baby_ind") == "N")
        .filter((F.col("sushrg") != "PB03Z") | F.col("sushrg").isNull())
        .filter(~((F.col("tretspef") == "424") & (F.col("epitype") == "3")))
        #
        .withColumnRenamed("person_id_deid", "person_id")
    ) 

    # for exclusions need column for primary diagnoses
    df_primary_diagnosis = (
        spark.read.table(table_names.hes_apc_diagnoses)
        .filter(F.col("diag_order") == 1)
        .withColumnRenamed("diagnosis", "primary_diagnosis")
        )

    prior = apply_readmissions_exclusions(
        prior
        .join(df_primary_diagnosis, ["epikey", "fyear", "procode3"], "left")
    ).alias("prior")

    join_condition = [
        F.col("readm.person_id") == F.col("prior.person_id"),
        F.col("readm.epikey") != F.col("prior.epikey"),
        (F.col("readm.admidate") > F.col("prior.admidate"))
        | (F.col("readm.disdate") > F.col("prior.disdate")),
        F.datediff(F.col("readm.admidate"), F.col("prior.disdate")).between(0, 29)
    ]

    return (
        readm.filter(F.col("admimeth").rlike("^2"))  
        .join(prior, join_condition, "semi")
        .withColumn("sample_rate", F.lit(1.0))
    )

Then have a function to prepare for DSR, which can be used for other activity avoidance TPMA:

In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days() # can swap this out for other TPMA function
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

Using a decorator, the function for direct standardised rates is added:

In [0]:
@directly_standardise
def get_tpma_dsr(spark, geography_column):
    return prepare_tpma_for_dsr(spark, geography_column)

Then get the data:

In [0]:
readmissions_within_30_days = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
)

# Existing TPMA

Spell level info for existing TPMA are here:

In [0]:
apc_mitigators = spark.table("udal_lake_mart.newhospitalprogramme.default_apc_mitigators")
apc = spark.table("udal_lake_mart.newhospitalprogramme.default_apc") # for spell info

apc_mitigators.limit(10).display()

The age and sex standardised rates that go into the TPMA explorer - `inputs_data` - are here:

In [0]:
a = spark.read.parquet("/Volumes/udal_lake_mart/newhospitalprogramme/files/inputs_data/dev/lad23cd/rates.parquet")

a.limit(10).display()

# Splitting by POD and age group

In the chunk below:
- I copied `readmissions_within_30_days` from above.
- Added a filter for POD to `prior`.
- Wrangled a column for `age` in `prior` using code from: https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py.
- Added a filter for age group to `prior`.
- Amended the join between `readm` and `prior` to account for some DQ issues with `age`.

This was to create a function I could then reuse for different `pod_filter` and `age_filter`.

In [0]:
def get_readmissions_within_30_days_pod(pod_filter, age_filter):
    spark = get_spark()

    readm = apply_readmissions_exclusions(
        nhp_apc
        # joining back to hes_apc to get disdest column - may be better to add disdest column to nhp_apc in longer run
        .join(
            spark.read.table(table_names.hes_apc).select(F.col("epikey"), F.col("fyear"), F.col("procode3"), F.col("disdest")), 
            ["epikey", "fyear", "procode3"],
            "left"
            )
        ).alias("readm")

    # make sure to use full hes table - our nhp views filter on certain columns
    # (e.g. not all providers included)
    prior = (
        spark.read.table(table_names.hes_apc)
        .filter(F.col("last_episode_in_spell"))
        # only spells discharged on clinical advice
        .filter(F.col("dismeth").isin(1, 3))
        # remove well babies
        .filter(F.col("well_baby_ind") == "N")
        .filter((F.col("sushrg") != "PB03Z") | F.col("sushrg").isNull())
        .filter(~((F.col("tretspef") == "424") & (F.col("epitype") == "3")))
        .withColumnRenamed("person_id_deid", "person_id")
    ) 

    # for exclusions need column for primary diagnoses
    df_primary_diagnosis = (
        spark.read.table(table_names.hes_apc_diagnoses)
        .filter(F.col("diag_order") == 1)
        .withColumnRenamed("diagnosis", "primary_diagnosis")
        )

    prior = (
        apply_readmissions_exclusions(
            prior
            .join(df_primary_diagnosis, ["epikey", "fyear", "procode3"], "left")
        )
        # filter for elective or non elective
        .filter(pod_filter)
        # filter to age group, applying the age logic from https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py
        .withColumn(
            "age",
            F.when(
                (F.col("admiage") == 999) | F.col("admiage").isNull(),
                F.when(F.col("startage") > 7000, 0).otherwise(F.col("startage")),
            ).otherwise(F.col("admiage")),
        )
        .withColumn("age", F.when(F.col("age") > 90, 90).otherwise(F.col("age")))
        .filter(age_filter).alias("prior")
    )

    join_condition = [
        F.col("readm.person_id") == F.col("prior.person_id"),
        F.col("readm.epikey") != F.col("prior.epikey"),
        (F.col("readm.admidate") > F.col("prior.admidate"))
        | (F.col("readm.disdate") > F.col("prior.disdate")),
        F.datediff(F.col("readm.admidate"), F.col("prior.disdate")).between(0, 29),
        # because prior is filtered to age but there are DQ issues, have to make sure 
        # `readm.age` is within an acceptable range of `prior.age`, otherwise patient
        # is counted in multiple age groups. Since, looking at readmissions within 30
        # days, `reamd.age` should be between `prior.age` and `prior.age` plus the length of 
        # prior spell plus 1 year (to account for potential birthday during prior spell). But 
        # also there were many rows where the prior was 1 year more than the readmission and 
        # so those are kept too:
        F.col("readm.age").between(F.col("prior.age") - 1, F.col("prior.age") + F.col("prior.speldur") + 1) 
    ]

    return (
        readm.filter(F.col("admimeth").rlike("^2"))  
        .join(prior, join_condition, "semi")
        .withColumn("sample_rate", F.lit(1.0))
    )

Defining the filters for each POD and age group:

In [0]:
age_0_17 = F.col("age") < 18
age_18_64 = ((F.col("age") >= 18) & (F.col("age") < 65))
age_65_plus = F.col("age") >= 65

non_elective = ((F.col("admimeth").rlike("^2")) & (F.col("classpat") == 1))
elective = ((F.col("admimeth").rlike("^1")) & (F.col("classpat") == 1))

Creating functions to get the readmissions TPMA by each POD and age group:

In [0]:
def get_readmissions_within_30_days_elective_0_17():
    return get_readmissions_within_30_days_pod(elective, age_0_17)
    
def get_readmissions_within_30_days_elective_18_64():
    return get_readmissions_within_30_days_pod(elective, age_18_64)
    
def get_readmissions_within_30_days_elective_65_plus():
    return get_readmissions_within_30_days_pod(elective, age_65_plus)

def get_readmissions_within_30_days_non_elective_0_17():
    return get_readmissions_within_30_days_pod(non_elective, age_0_17)
    
def get_readmissions_within_30_days_non_elective_18_64():
    return get_readmissions_within_30_days_pod(non_elective, age_18_64)
    
def get_readmissions_within_30_days_non_elective_65_plus():
    return get_readmissions_within_30_days_pod(non_elective, age_65_plus)

Getting the data:

In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_elective_0_17()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_elective_0_17 = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("elective"))
    .withColumn("age_group", F.lit("0-17"))
)

In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_elective_18_64()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_elective_18_64 = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("elective"))
    .withColumn("age_group", F.lit("18-64"))
)

In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_elective_65_plus()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_elective_65_plus = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()    
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("elective"))
    .withColumn("age_group", F.lit("65_plus"))
)

In [0]:

def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_non_elective_0_17()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_non_elective_0_17 = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("non_elective"))
    .withColumn("age_group", F.lit("0-17"))
)


In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_non_elective_18_64()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_non_elective_18_64 = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("non_elective"))
    .withColumn("age_group", F.lit("18-64"))
)

In [0]:
def prepare_tpma_for_dsr(spark, geography_column):
    df = (        
        get_readmissions_within_30_days_non_elective_65_plus()
        .withColumn("type", F.lit("activity_avoidance"))
        .withColumn("strategy", F.lit("readmission_within_30_days"))
        .groupBy("fyear", "age", "sex", geography_column, "type", "strategy")
        .agg(F.count("fyear").alias("n"))
    )
   
    pop = get_population(spark, geography_column)

    df = complete_age_sex_rows(spark, df, geography_column)

    return df.join(pop, ["fyear", "age", "sex", geography_column], "left").filter(
        F.col("d") > 0
    )

readmissions_within_30_days_non_elective_65_plus = (  
    get_tpma_dsr(spark, "lad23cd")
    .orderBy("fyear", "lad23cd")
    .persist()
    .withColumn("status", F.lit("new"))
    .withColumn("pod", F.lit("non_elective"))
    .withColumn("age_group", F.lit("65_plus"))
)

# Saving data

In [0]:
readmissions_within_28_days = (
  spark.read.parquet("/Volumes/udal_lake_mart/newhospitalprogramme/files/inputs_data/dev/lad23cd/rates.parquet")
  .filter(F.col("strategy") == "readmission_within_28_days")
  .withColumn("status", F.lit("existing"))
  .withColumn("pod", F.lit("all"))
  .withColumn("age_group", F.lit("all"))
  .select(
      F.col("fyear"), 
      F.col("strategy"), 
      F.col("lad23cd"), 
      F.col("crude_rate"), 
      F.col("std_rate"), 
      F.col("denominator"), 
      F.col("status"), 
      F.col("pod"), 
      F.col("age_group")
      )
)

In [0]:
readmissions_tpma_rates = (
  readmissions_within_30_days
  .withColumn("status", F.lit("new"))
  .withColumn("pod", F.lit("all"))
  .withColumn("age_group", F.lit("all"))
  .union(readmissions_within_28_days)
  .union(readmissions_within_30_days_elective_0_17)
  .union(readmissions_within_30_days_elective_18_64)
  .union(readmissions_within_30_days_elective_65_plus)
  .union(readmissions_within_30_days_non_elective_0_17)
  .union(readmissions_within_30_days_non_elective_18_64)
  .union(readmissions_within_30_days_non_elective_65_plus)
  .filter(F.col("fyear") >= 201516)
)

In [0]:
# Lakemart where we're saving outputs
lakeNameSU = "udalstdataanalysisprod.dfs.core.windows.net"
containerNameSU = "analytics-projects"
tpma_review_path = "abfss://"+containerNameSU+"@"+lakeNameSU+"/StrategyUnit/StrategyUnit/analytics_modelling/nhp_tpma_review/2026_08_readmissions/"

# Write parquet file
readmissions_tpma_rates.write.mode("overwrite").parquet(tpma_review_path + "readmissions_rates.parquet")

# Getting age and sex data

Defining a function so can easily repeat for existing, new and subgroups. Then collating all data into one file to save.

In [0]:
def get_readmissions_by_age_and_sex(data, pod, age_group, status):    
    return(data
        .filter(F.col("fyear") == 202425)
        .withColumn(
            "age_band",
            F.when(F.col("age") < 5, "0-4")
            .when(F.col("age") < 10, "5-9")
            .when(F.col("age") < 15, "10-14")
            .when(F.col("age") < 20, "15-19")
            .when(F.col("age") < 25, "20-24")
            .when(F.col("age") < 30, "25-29")
            .when(F.col("age") < 35, "30-34")
            .when(F.col("age") < 40, "35-39")
            .when(F.col("age") < 45, "40-44")
            .when(F.col("age") < 50, "45-49")
            .when(F.col("age") < 55, "50-54")
            .when(F.col("age") < 60, "55-59")
            .when(F.col("age") < 65, "60-64")
            .when(F.col("age") < 70, "65-69")
            .when(F.col("age") < 75, "70-74")
            .when(F.col("age") < 80, "75-79")
            .when(F.col("age") < 85, "80-84")
            .when(F.col("age") < 90, "85-89")
            .when(F.col("age") >= 90, "90+")
            .otherwise("NULL")
            )
    .groupBy("age_band", "sex")
    .agg(F.count("fyear").alias("n"))
    .withColumn("status", F.lit(status))
    .withColumn("pod", F.lit(pod))
    .withColumn("age_group", F.lit(age_group))
    )

In [0]:
readmissions_tpma_age_sex = (
  get_readmissions_by_age_and_sex(
      apc_mitigators
      .filter(F.col("strategy") == "readmission_within_28_days")
      .join(apc, ["epikey", "fyear", "provider"], "left"), 
      "all", "all", "existing"
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days(), "all", "all", "new"
          )
      )
    .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_elective_0_17(), "elective", "0-17", "new"
          )
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_elective_18_64(), "elective", "18-64", "new"
          )
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_elective_65_plus(), "elective", "65_plus", "new"
          )
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_non_elective_0_17(), "non_elective", "0-17", "new"
          )
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_non_elective_18_64(), "non_elective", "18-64", "new"
          )
      )
  .union(
      get_readmissions_by_age_and_sex(
          get_readmissions_within_30_days_non_elective_65_plus(), "non_elective", "65_plus", "new"
          )
      )
)

In [0]:
# Write parquet file
readmissions_tpma_age_sex.write.mode("overwrite").parquet(tpma_review_path + "readmissions_age_sex.parquet")

# Getting top n data

Getting top n primary diagnoses or primary procedure. Function get_top_n() can be used for any TPMA, for either primary_diagnoses or primary_procedure and for any n.

In [0]:
def get_top_n(data, type, n, status, pod, age_group):
    top_n = (
        data
        .filter((F.col("fyear") == 202425) & (~F.col(type).isNull()))
        .withColumn("primary_diagnosis", F.left(F.col("primary_diagnosis"), F.lit(3)))
        .withColumn("primary_procedure", F.left(F.col("primary_procedure"), F.lit(3)))
        .groupBy(type)
        .agg(F.count("fyear").alias("n"))
        .withColumn("perc", F.col("n") / F.sum("n").over(Window.partitionBy()))
        .sort(F.col("n").desc())
        .limit(n)        
        .withColumn("status", F.lit(status))
        .withColumn("pod", F.lit(pod))
        .withColumn("age_group", F.lit(age_group))
        )

    return(top_n)

Defining a function so can easily repeat for existing, new and subgroups. Then collating all data into one file to save.

In [0]:
def get_top_n_readmissions(type, n):
    top_n = (
        get_top_n(
        apc_mitigators
            .filter(F.col("strategy") == "readmission_within_28_days")
            .join(apc, ["epikey", "fyear", "provider"], "left"),
            type, n, "existing", "all", "all"  
        )
        .union(
            get_top_n(
                get_readmissions_within_30_days(), 
                type, n, "new", "all", "all"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_elective_0_17(), 
                type, n, "new", "elective", "0-17"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_elective_18_64(), 
                type, n, "new", "elective", "18-64"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_elective_65_plus(), 
                type, n, "new", "elective", "65_plus"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_non_elective_0_17(), 
                type, n, "new", "non_elective", "0-17"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_non_elective_18_64(), 
                type, n, "new", "non_elective", "18-64"
                )
            )
        .union(
            get_top_n(
                get_readmissions_within_30_days_non_elective_65_plus(), 
                type, n, "new", "non_elective", "65_plus"
                )
            )
        )
    
    return(top_n)

In [0]:
readmissions_top_n = (
    get_top_n_readmissions("primary_diagnosis", 6).withColumn("type", F.lit("diagnosis"))
    .union(
        get_top_n_readmissions("primary_procedure", 6).withColumn("type", F.lit("procedure"))
    )
    .withColumnRenamed("primary_diagnosis", "code")
)

# Write parquet file
readmissions_top_n.write.mode("overwrite").parquet(tpma_review_path + "readmissions_top_n.parquet")

# Getting overlaps data

Defining a function so can easily repeat for existing, new and subgroups. Then collating all data into one file to save.

In [0]:
other_existing_tpma = (
    apc_mitigators
    .filter((F.col("fyear") == 202425) & (F.col("strategy") != "readmission_within_28_days"))
    )

def get_overlap_with_other_tpma(data, status, pod, age_group):
    total = data.filter(F.col("fyear") == 202425).count()

    overlap = (
        data
        .filter(F.col("fyear") == 202425) 
        .select(F.col("provider"), F.col("epikey"))
        .join(other_existing_tpma, ["epikey", "provider"], "inner")
        .groupBy("strategy")
        .agg(F.sum("sample_rate").alias("n"))   
        .withColumn("perc", F.col("n") / total)     
        .withColumn("status", F.lit(status))
        .withColumn("pod", F.lit(pod))
        .withColumn("age_group", F.lit(age_group))
    )

    return(overlap)

In [0]:
readmissions_overlaps = (
    get_overlap_with_other_tpma(
        apc_mitigators
        .filter(F.col("strategy") == "readmission_within_28_days"),
        "existing", "all", "all"
        )    
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days(), "new", "all", "all"))
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_elective_0_17(), "new", "elective", "0-17"))
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_elective_18_64(), "new", "elective", "18-64"))    
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_elective_65_plus(), "new", "elective", "65_plus"))
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_non_elective_0_17(), "new", "non_elective", "0-17"))
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_non_elective_18_64(), "new", "non_elective", "18-64"))    
    .union(get_overlap_with_other_tpma(get_readmissions_within_30_days_non_elective_65_plus(), "new", "non_elective", "65_plus"))
)

#Write parquet file
readmissions_overlaps.write.mode("overwrite").parquet(tpma_review_path + "readmissions_overlaps.parquet")

# Percentage of readmissions

Numerator = number of readmissions following an index spell in FY. \
Denominator = number of index spells in FY.

Would use FY = 202425, but `nhp_apc` only has data from completed financial years. So any index spells in March 25 with readmissions in April 25 would not be picked up. So latest FY can use is 202324 for this.

For the new TPMA:

In [0]:
spark = get_spark()

readm = (
    apply_readmissions_exclusions(
    nhp_apc
    # joining back to hes_apc to get disdest column - may be better to add disdest column to nhp_apc in longer run
    .join(
        spark.read.table(table_names.hes_apc).select(F.col("epikey"), F.col("fyear"), F.col("procode3"), F.col("disdest")), 
        ["epikey", "fyear", "procode3"],
        "left"
        )
        )
    .filter(F.col("admimeth").rlike("^2"))
    .withColumn("is_readmission", F.lit(1.0))
    .select(F.col("person_id"), F.col("epikey"), (F.col("admidate")), (F.col("disdate")), F.col("is_readmission"))
    .alias("readm")
    )

# make sure to use full hes table - our nhp views filter on certain columns
# (e.g. not all providers included)
prior = (
    spark.read.table(table_names.hes_apc)
    .filter(F.col("last_episode_in_spell"))
    # only spells discharged on clinical advice
    .filter(F.col("dismeth").isin(1, 3))
    # remove well babies
    .filter(F.col("well_baby_ind") == "N")
    .filter((F.col("sushrg") != "PB03Z") | F.col("sushrg").isNull())
    .filter(~((F.col("tretspef") == "424") & (F.col("epitype") == "3")))
    #
    .withColumnRenamed("person_id_deid", "person_id")
) 

# for exclusions need column for primary diagnoses
df_primary_diagnosis = (
    spark.read.table(table_names.hes_apc_diagnoses)
    .filter(F.col("diag_order") == 1)
    .withColumnRenamed("diagnosis", "primary_diagnosis")
    )

prior = apply_readmissions_exclusions(
    prior
    .join(df_primary_diagnosis, ["epikey", "fyear", "procode3"], "left")
).alias("prior")

join_condition = [
    F.col("readm.person_id") == F.col("prior.person_id"),
    F.col("readm.epikey") != F.col("prior.epikey"),
    (F.col("readm.admidate") > F.col("prior.admidate"))
    | (F.col("readm.disdate") > F.col("prior.disdate")),
    F.datediff(F.col("readm.admidate"), F.col("prior.disdate")).between(0, 29)
]

prior_with_readm_2324 = (
    prior 
    .filter(F.col("prior.fyear") == "202324")
    .join(readm, join_condition, "left")
    .withColumn("sample_rate", F.lit(1.0))
    .select(F.col("prior.epikey"), F.col("fyear"), F.col("is_readmission"), F.col("sample_rate"))
    .distinct()
)

prior_with_readm_2324_new = (
    prior_with_readm_2324
    .agg(F.sum("is_readmission").alias("n_numerator"), F.sum("sample_rate").alias("n_denominator"))    
    .withColumn("status", F.lit("new"))
)

For the existing TPMA:

In [0]:

spark = get_spark()

readm = (
    nhp_apc
    .withColumn("is_readmission", F.lit(1.0))
    .select(F.col("person_id"), F.col("epikey"), (F.col("admidate")), (F.col("disdate")), F.col("is_readmission"))
    ).alias("readm")

prior = (
    spark.read.table(table_names.hes_apc)
    .filter(F.col("last_episode_in_spell"))
    # remove well babies
    .filter(F.col("well_baby_ind") == "N")
    .filter((F.col("sushrg") != "PB03Z") | F.col("sushrg").isNull())
    .filter(~((F.col("tretspef") == "424") & (F.col("epitype") == "3")))
    .withColumnRenamed("person_id_deid", "person_id")
    .alias("prior")
)

join_condition = [
    F.col("readm.person_id") == F.col("prior.person_id"),
    F.col("readm.epikey") != F.col("prior.epikey"),
    (F.col("readm.admidate") > F.col("prior.admidate"))
    | (F.col("readm.disdate") > F.col("prior.disdate")),
    F.datediff(F.col("readm.admidate"), F.col("prior.disdate")).between(0, 28),
]

prior_with_readm_2324 = (
    prior 
    .filter(F.col("prior.fyear") == "202324")
    .join(readm, join_condition, "left")
    .withColumn("sample_rate", F.lit(1.0))
    .select(F.col("prior.epikey"), F.col("fyear"), F.col("is_readmission"), F.col("sample_rate"))
    .distinct()
)

prior_with_readm_2324_existing = (
    prior_with_readm_2324
    .agg(F.sum("is_readmission").alias("n_numerator"), F.sum("sample_rate").alias("n_denominator"))    
    .withColumn("status", F.lit("existing"))
)

In [0]:
readmissions_percentage = (
    prior_with_readm_2324_new 
    .union(prior_with_readm_2324_existing)
)

#Write parquet file
readmissions_percentage.write.mode("overwrite").parquet(tpma_review_path + "readmissions_percentage.parquet")